# Velocity-weakening Workbench

Use this notebook to inspect an existing run or launch a new one without typing a long CLI command. The notebook calls the same `velocity_weakening_tatva.py` driver you have been using in the terminal, but exposes the most common settings as Python variables.


## 1. Imports and Helper Functions

Run this cell first. It loads the helper utilities from `src/velocity_weakening_notebook_helpers.py`.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from velocity_weakening_notebook_helpers import (
    DEFAULT_MU_DISP_Y_POINTS,
    REPO_ROOT,
    RUNS_ROOT,
    build_driver_command,
    default_case_config,
    launch_case,
    list_runs,
    load_summary_for_run,
    print_runs,
    resolve_run,
    run_paths,
)

print(f"Repo root: {REPO_ROOT}")
print(f"Runs root: {RUNS_ROOT}")
print(f"Default mu-disp y-points: {DEFAULT_MU_DISP_Y_POINTS}")


## 2. Inspect an Existing Run

You can locate a run in three ways:

- `RUN_NUMBER`: for example `5`
- `RUN_SUFFIX`: the part after `0005_...`
- `RUN_NAME`: the full folder name

Usually `RUN_NUMBER` is enough.


In [ ]:
RUN_NUMBER = 5
RUN_SUFFIX = "lockedge-normal40ms-ramp20ms-hold20ms-shear0p530994ms-shearx2p5-mesh2mm"
RUN_NAME = None
LIST_LIMIT = 10


In [ ]:
print_runs(limit=LIST_LIMIT)

run_dir = resolve_run(
    run_number=RUN_NUMBER,
    run_suffix=RUN_SUFFIX,
    run_name=RUN_NAME,
)
summary_payload = load_summary_for_run(run_dir)
paths = run_paths(run_dir)

print(f"\nResolved run: {run_dir.name}")
print(json.dumps(paths, indent=2))

summary = summary_payload["summary"]
important = {
    "mesh_size": summary.get("mesh_size"),
    "dt": summary.get("dt"),
    "saved_frames": summary.get("saved_frames"),
    "pressure_frames_saved": summary.get("pressure_frames_saved"),
    "shear_frames_saved": summary.get("shear_frames_saved"),
    "final_max_slip": summary.get("final_max_slip"),
    "final_avg_tau": summary.get("final_avg_tau"),
    "final_avg_sigma_n": summary.get("final_avg_sigma_n"),
}
important


## 3. Prepare a New Case

Edit only the values you care about. Everything else falls back to the current default workflow:

- CPU execution through `conda run -n tatva`
- `2D` or `3D` selection with configurable thickness
- PDF plots
- 4K `von Mises` and `sigma_xy` animations
- four-panel `mu-disp` plot at `y = 125, 250, 375, 450 mm`


In [ ]:
NEW_CASE = default_case_config()

# Common fields to edit
NEW_CASE["run_label"] = "lockedge-normal40ms-ramp20ms-hold20ms-shear0p530994ms-shearx2p5-mesh2mm"
NEW_CASE["dimension"] = 2
NEW_CASE["thickness"] = 0.0
NEW_CASE["mesh_size"] = 2.0
NEW_CASE["normal_phase_time"] = 0.04
NEW_CASE["normal_ramp_time"] = 0.02
NEW_CASE["shear_phase_time"] = 0.000530994
NEW_CASE["shear_scale"] = 2.5
NEW_CASE["frames_per_phase"] = 4800
NEW_CASE["shear_frames_per_phase"] = 28800
NEW_CASE["num_threads"] = 8

# Advanced fields (uncomment and edit if needed)
# NEW_CASE["dimension"] = 3
# NEW_CASE["thickness"] = 50.0
# NEW_CASE["normal_penalty"] = 38310.0
# NEW_CASE["tangential_penalty"] = 3831.0
# NEW_CASE["skip_animation"] = True
# NEW_CASE["skip_shear_stress_animation"] = True
# NEW_CASE["animation_workers"] = 8
# NEW_CASE["mu_disp_y_points"] = [125.0, 250.0, 375.0, 450.0]

NEW_CASE


In [ ]:
cmd = build_driver_command(NEW_CASE)
print("Command preview:\n")
print(" ".join(cmd))


## 4. Launch the New Case

Set `RUN_NEW_CASE = True` only when you are ready. This will run the full simulation and post-processing pipeline.


In [ ]:
RUN_NEW_CASE = False

if RUN_NEW_CASE:
    launch_case(NEW_CASE, stream_output=True)
else:
    print("Set RUN_NEW_CASE = True when you want to launch the case.")
